# Reinforcement Learning with GRPO



In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch

from mi.extraction import extract_final_candidate
from mi.gpu import get_device
from mi.lm.model_loader import load_model_and_tokenizer

# device = get_device()
device = torch.device("cpu")

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

✓ downloads/qwen3-0.6B-base.pth already up-to-date


In [3]:
# Verify model loaded correctly
from mi.lm.prompts import render_prompt
from mi.lm.generate import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache
)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

torch.manual_seed(0)
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9,
)
# print(response)

 \boxed{58}

In [4]:
# Load MATH training subset
from mi.dataloaders.math import load_math_train

math_train = load_math_train()
print(f"Dataset size: {len(math_train)}")

Dataset size: 12000


In [5]:
from pprint import pprint
pprint(math_train[4])

{'answer': '6',
 'level': 'Level 3',
 'problem': 'Sam is hired for a 20-day period. On days that he works, he earns '
            '$\\$$60. For each day that he does not work, $\\$$30 is '
            'subtracted from his earnings. At the end of the 20-day period, he '
            'received $\\$$660. How many days did he not work?',
 'solution': 'Call $x$ the number of days Sam works and $y$ the number of days '
             'he does not. We can set up the following system of equations to '
             'represent the given information: \\begin{align*}\n'
             'x+y &= 20 \\\\\n'
             '60x - 30y &= 660 \\\\\n'
             '\\end{align*} The first equation represents the total number of '
             'days Sam works, and the second equation represents his total '
             'profit. Solving for $x$ in the first equation yields $x = 20 - '
             'y$. Substituting into the second equation gives $60(20-y) - 30y '
             '= 660$. Canceling a factor of $10$ an

## Sampling Rollouts

In [6]:
from mi.lm.generate import sample_response

torch.manual_seed(5)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

token_ids, prompt_len, answer_text = sample_response(
    model,
    tokenizer=tokenizer,
    prompt=prompt,
    device=device,
    max_new_tokens=512,
    temperature=0.9,
    top_p=0.9,
)

print(answer_text)

 Let's solve the problem step by step.

**Given:**
\[
\text{Half the value of } 3x - 9 \text{ is } x + 37.
\]

**Step 1: Translate the statement into an equation.**
\[
\frac{1}{2} (3x - 9) = x + 37
\]

**Step 2: Eliminate the fraction by multiplying both sides by 2.**
\[
3x - 9 = 2(x + 37)
\]

**Step 3: Distribute the 2 on the right side.**
\[
3x - 9 = 2x + 74
\]

**Step 4: Subtract \(2x\) from both sides to get the \(x\)-terms on one side.**
\[
3x - 2x - 9 = 74
\]
\[
x - 9 = 74
\]

**Step 5: Add 9 to both sides to solve for \(x\).**
\[
x = 74 + 9
\]
\[
x = 83
\]

**Final Answer:**
\[
\boxed{83}
\]<|endoftext|>


In [7]:
# demo rollouts
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

## Calculating Rewards

In [8]:
from mi.lm.scoring import reward_rlvr

rollout_rewards = []

for answer in rollouts:
    reward = reward_rlvr(answer_text=answer, ground_truth="83")
    print(f"Answer: {answer!r}")
    print(f"Reward: {reward!r}\n")
    rollout_rewards.append(reward)

Answer: '\\boxed{83}'
Reward: 1.0

Answer: 'The correct answer is \\boxed{83}'
Reward: 1.0

Answer: 'The final answer is 83'
Reward: 0.0

Answer: 'We get \\boxed{38}'
Reward: 0.0



## Preparing Learning Signals via Advantages

In [9]:
rewards = torch.tensor(rollout_rewards, device=device)
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)
print(rewards)
print(advantages)

tensor([1., 1., 0., 0.])
tensor([ 0.8659,  0.8659, -0.8659, -0.8659])


In [10]:
partial_credit_rewards = []
for answer in rollouts:
    reward = reward_rlvr(answer_text=answer, ground_truth="83", partial_credit=0.5)
    partial_credit_rewards.append(reward)


partial_rewards = torch.tensor(partial_credit_rewards, device=device)
partial_advantages = (partial_rewards - partial_rewards.mean()) / (partial_rewards.std() + 1e-4)

print(partial_rewards)
print(partial_advantages)

tensor([1.0000, 1.0000, 0.5000, 0.0000])
tensor([ 0.7832,  0.7832, -0.2611, -1.3053])


## Scoring Rollouts with Sequence Log Probabilities

In [11]:
from mi.lm.scoring import avg_logprob_answer

avg_logprob_val = avg_logprob_answer(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    answer=answer_text,
    device=device,
)
print(avg_logprob_val)

tensor(-0.0613, dtype=torch.bfloat16)


In [12]:
sequence_logprob_val = avg_logprob_val * len(tokenizer.encode(answer_text))
print(sequence_logprob_val)

tensor(-16.2500, dtype=torch.bfloat16)


In [13]:
from mi.lm.scoring import sequence_logprob_draft

sequence_logprob_draft = sequence_logprob_draft(
    model=model,
    token_ids=token_ids,
    prompt_len=prompt_len,
)
print(sequence_logprob_draft)

tensor(-16.2998, grad_fn=<SumBackward0>)


In [14]:
from mi.lm.scoring import sequence_logprob


sequence_logprob_result = sequence_logprob(
    model=model,
    token_ids=token_ids,
    prompt_len=prompt_len,
)

print(sequence_logprob_result)

tensor(-16.2998, grad_fn=<SumBackward0>)


In [15]:
rollout_logps = []

for text in rollouts:
    token_ids = tokenizer.encode(prompt + " " + text)
    logprob = sequence_logprob(
        model=model,
        token_ids=torch.tensor(token_ids, device=device),
        prompt_len=prompt_len,
    )

    print(f"Answer:    {text}")
    print(f"Logprob:   {logprob.item():.4f}\n")
    rollout_logps.append(logprob)


Answer:    \boxed{83}
Logprob:   -7.9243

Answer:    The correct answer is \boxed{83}
Logprob:   -20.1546

Answer:    The final answer is 83
Logprob:   -16.6130

Answer:    We get \boxed{38}
Logprob:   -23.3677



## From Advantages to Policy Updates via the GRPO Loss
compute the policy gradient loss by multiplying each rollout’s sequence-level logprob by its corresponding advantage, taking the mean across rollouts, and applying a negative sign. Because PyTorch optimizers are defined to minimize a loss, objectives that are naturally written as maximization problems must be sign-flipped.

__Sebastian Raschka. Build_a_Reasoning_Model_(From_Scratch) (Function). Kindle Edition.__

In [16]:
logps = torch.stack(rollout_logps)
pg_loss = -(advantages.detach() * logps).mean()
print(logps)
print(pg_loss)

tensor([ -7.9243, -20.1546, -16.6130, -23.3677], grad_fn=<StackBackward0>)
tensor(-2.5764, grad_fn=<NegBackward0>)


## A Single GRPO Function

In [17]:
from mi.rl.grpo import compute_grpo_loss


torch.manual_seed(123)

stats = compute_grpo_loss(
    model=model,
    tokenizer=tokenizer,
    example=math_train[4],
    device=device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
)

pprint(stats)

{'advantages': [0.0, 0.0],
 'loss': -0.0,
 'loss_tensor': tensor(-0., grad_fn=<NegBackward0>),
 'pg_loss': -0.0,
 'rewards': [0.0, 0.0],
 'samples': [{'gen_len': 4, 'reward': 0.0, 'text': ' 14<|endoftext|>'},
             {'gen_len': 256,
              'reward': 0.0,
              'text': ' 4\n'
                      '\n'
                      "To solve the problem, let's break it down step by "
                      'step:\n'
                      '\n'
                      '1. **Define Variables:**\n'
                      '   - Let \\( x \\) be the number of days Sam works.\n'
                      '   - Then, the number of days he does not work is \\( '
                      '20 - x \\).\n'
                      '\n'
                      '2. **Set Up the Earnings Equation:**\n'
                      '   - For each day he works, he earns \\$60.\n'
                      '   - For each day he does not work, he loses \\$30.\n'
                      '   - His total earnings are \\$660.

## GRPO Training Loop

In [18]:
import time
from pathlib import Path

def train_rlvr_grpo(
        model,
        tokenizer,
        math_data,
        device,
        steps=None,
        num_rollouts=2,
        max_new_tokens=256,
        temperature=0.8,
        top_p=0.9,
        lr=1e-5,
        checkpoint_every=50,
        checkpoint_dir=".",
        csv_log_path=None,
):
    if steps is None:
        steps = len(math_data)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    current_step = 0

    if csv_log_path is None:
        timestamp = time.strftime("%Y%m%d-%H%M%S")
        csv_log_path = f"train_rlvr_grpo_{timestamp}.csv"

    csv_log_path = Path(csv_log_path)

    try:
        for step in range(steps):
            optimizer.zero_grad()

            current_step = step + 1
            example = math_data[step % len(math_data)]

            stats = compute_grpo_loss(
                model=model,
                tokenizer=tokenizer,
                example=example,
                device=device,
                num_rollouts=num_rollouts,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
            )

            stats["loss_tensor"].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            reward_avg = torch.tensor(stats["rewards"]).mean().item()
            step_tokens = sum(
                sample["gen_len"] for sample in stats["samples"]
            )
            avg_response_len = (
                step_tokens / len(stats["samples"]) if stats["samples"] else 0.0
            )
            append_csv_metrics(
                csv_log_path, current_step, steps,
                stats["loss"], reward_avg, avg_response_len,
            )
            print(
                f"[Step {current_step}/{steps}] "
                f"loss={stats['loss']:.4f} "
                f"reward_avg={reward_avg:.3f} "
                f"avg_response_len={avg_response_len:.1f}"
            )

            if checkpoint_every and current_step % checkpoint_every == 0:
                ckpt_path = save_checkpoint(
                    model=model,
                    checkpoint_dir=checkpoint_dir,
                    step=current_step,
                )
                print(f"Saved checkpoint at {ckpt_path}")
    except KeyboardInterrupt:
        ckpt_path = save_checkpoint(
            model=model,
            checkpoint_dir=checkpoint_dir,
            step=max(1, current_step),
            suffix="interrupt",
        )
        print(f"\nKeyboardInterrupt. Saved checkpoint at {ckpt_path}")
        return model

    return model


def save_checkpoint(model, checkpoint_dir, step, suffix=""):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    suffix = f"-{suffix}" if suffix else ""
    ckpt_path = (
        checkpoint_dir /
        f"qwen3-0.6B-rlvr-grpo-step{step:05d}{suffix}.pth"
    )
    torch.save(model.state_dict(), ckpt_path)
    return ckpt_path


def append_csv_metrics(
        csv_log_path,
        step_idx,
        total_steps,
        loss,
        reward_avg,
        avg_response_len,
):
    if not csv_log_path.exists():
        csv_log_path.write_text(
            "step,total_steps,loss,reward_avg,avg_response_len\n",
            encoding="utf-8",
        )

    with csv_log_path.open("a", encoding="utf-8") as f:
        f.write(
            f"{step_idx},{total_steps},{loss:6f},{reward_avg:6f},{avg_response_len:6f}\n"
        )


In [19]:
device = get_device()
model.to(device)

torch.manual_seed(0)

train_rlvr_grpo(
    model=model,
    tokenizer=tokenizer,
    math_data=math_train,
    device=device,
    steps=50,
    num_rollouts=4,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=5,
    checkpoint_dir=".",
)

Using Apple Silicon GPU (MPS)
[Step 1/50] loss=-0.0000 reward_avg=0.000 avg_response_len=6.2
[Step 2/50] loss=-0.0000 reward_avg=0.000 avg_response_len=6.8
[Step 3/50] loss=9.2032 reward_avg=0.500 avg_response_len=56.0
[Step 4/50] loss=2.9223 reward_avg=0.250 avg_response_len=56.5
[Step 5/50] loss=-1.3587 reward_avg=0.375 avg_response_len=185.8
Saved checkpoint at qwen3-0.6B-rlvr-grpo-step00005.pth
[Step 6/50] loss=2.6966 reward_avg=0.500 avg_response_len=347.8
[Step 7/50] loss=-0.0000 reward_avg=0.000 avg_response_len=512.0
[Step 8/50] loss=3.9098 reward_avg=0.750 avg_response_len=188.0
[Step 9/50] loss=-0.0000 reward_avg=0.000 avg_response_len=468.8
[Step 10/50] loss=-0.0000 reward_avg=0.000 avg_response_len=465.8
Saved checkpoint at qwen3-0.6B-rlvr-grpo-step00010.pth
[Step 11/50] loss=-0.0000 reward_avg=1.000 avg_response_len=328.5
[Step 12/50] loss=-0.0000 reward_avg=0.000 avg_response_len=351.2
[Step 13/50] loss=-0.0000 reward_avg=0.000 avg_response_len=253.8
[Step 14/50] loss=-0.

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [ ]:
# Load saved checkpoint file
model.load_state_dict(torch.load("qwen3-0.6B-rlvr-grpo-step00050.pth"))
